## `POPSRegressionEllipse`: ellipsoid posteriors by direct optimization

Comparing `BayesianRidge` (epistemic only), `POPSRegression`
(sampling-based POPS hypercube) and `POPSRegressionEllipse`
(uniform-ellipsoid posterior fit by direct minimization of the
generalization-error objective) on a misspecified polynomial surrogate
fit to low-noise data.

The ellipsoid bounds track the POPS hypercube bounds, but are obtained by
an interior-point optimization of the exact projected-ball pushforward
likelihood — no posterior sampling is involved.

In [ ]:
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import PolynomialFeatures
from popsregression import POPSRegression, POPSRegressionEllipse

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def target_function(x):
    return (x**3 + 0.01 * x**4) * 0.1 + np.sin(x) * x * 10.0


def generate_data(N):
    x_train = np.sort(
        np.append(np.random.uniform(-1, 1, N), np.linspace(-1, 1, 2)) * 10
    )
    x_dense = np.linspace(-1.1, 1.1, 51) * 10
    y_dense = target_function(x_dense)

    p = PolynomialFeatures(degree=4, include_bias=True)
    X_train = p.fit_transform(x_train.reshape(-1, 1))
    X_dense = p.fit_transform(x_dense.reshape(-1, 1))
    y_train = target_function(x_train)
    return X_train, x_train, y_train, X_dense, x_dense, y_dense


def plot_panel(ax, x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=None, y_min=None):
    if y_max is not None and y_min is not None:
        ax.fill_between(x_dense, y_min, y_max, alpha=0.2, facecolor="0.5",
                        label="max/min")
    else:
        ax.fill_between(x_dense, y_pred - 4 * y_std, y_pred + 4 * y_std,
                        alpha=0.2, facecolor="0.5", label=r"$\pm4\sigma$")
    ax.fill_between(x_dense, y_pred - 2 * y_std, y_pred + 2 * y_std,
                    alpha=0.5, facecolor="C1", label=r"$\pm2\sigma$")
    ax.plot(x_dense, y_pred, "C1-", lw=4)
    ax.plot(x_train, y_train, "b.", label="Train")
    ax.plot(x_dense, y_dense, "k-")

### BayesianRidge vs POPS Hypercube vs POPS Ellipse

Fitting a quartic polynomial (P=5) to a complex oscillatory function at
N = 10, 50, 500 training points. BayesianRidge epistemic uncertainty
vanishes with more data; both POPS variants maintain honest uncertainty
where the polynomial deviates from the truth. The ellipse bounds follow
the hypercube bounds while being produced by direct optimization.

In [ ]:
np.random.seed(42)

titles = ["Bayesian Ridge", "POPS Hypercube", "POPS Ellipse"]
fig, axs = plt.subplots(3, 3, figsize=(8, 7), sharex=True, sharey=True)
N_array = [10, 50, 500]

for i, N in enumerate(N_array):
    X_train, x_train, y_train, X_dense, x_dense, y_dense = generate_data(N)

    bay = BayesianRidge(fit_intercept=False)
    hyc = POPSRegression(leverage_percentile=0.0, posterior="hypercube")
    ell = POPSRegressionEllipse(random_state=0)

    bay.fit(X_train, y_train)
    hyc.fit(X_train, y_train)
    ell.fit(X_train, y_train)

    # BayesianRidge - epistemic only (no aleatoric alpha_)
    b_pred = bay.predict(X_dense, return_std=False)
    b_std = np.sqrt(np.sum(np.dot(X_dense, bay.sigma_) * X_dense, axis=1))
    plot_panel(axs[0, i], x_dense, y_dense, x_train, y_train, b_pred, b_std)

    # POPS Hypercube (sampling-based)
    y_pred, y_std, y_max, y_min = hyc.predict(
        X_dense, return_std=True, return_bounds=True
    )
    plot_panel(axs[1, i], x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=y_max, y_min=y_min)

    # POPS Ellipse (direct optimization; bounds = pushforward support)
    y_pred, y_std, y_max, y_min = ell.predict(
        X_dense, return_std=True, return_bounds=True
    )
    plot_panel(axs[2, i], x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=y_max, y_min=y_min)

    axs[0, i].set_title(f"N = {N}")

for j, title in enumerate(titles):
    axs[j, 0].set_ylim(-250, 250)
    axs[j, 0].set_xlim(-10, 10)
    axs[j, 1].legend(fontsize=9, loc="lower center")
    axs[j, 0].set_ylabel(title, fontsize=13)

plt.tight_layout()
plt.show()

### Observations

- **N/P = 2**: all methods show wide uncertainty; both POPS variants
  cover the training data. The ellipse bounds are noticeably *tighter*
  than the hypercube at small N: the ellipsoid is the **minimum-width
  shape covering the training residuals** (an empirical-risk minimum),
  while the hypercube min/max bounds are the support of a box built
  from the raw pointwise-correction spread — un-optimized, and
  geometrically wider than an ellipsoid along the same directions.
- **N/P = 10 and 100**: BayesianRidge epistemic uncertainty collapses,
  while the hypercube and ellipse bounds persist, reflecting the
  structural inability of the quartic to match the oscillatory target.
- The ellipse bounds come from the optimized ellipsoid support
  $\text{mean} \pm \sqrt{x^\top B x + \delta^2}$ rather than from
  posterior samples, and every training point is covered by
  construction (`coverage_fraction_ == 1`).
- The ellipse **mean** deliberately differs from BayesianRidge: its
  stationarity condition is a heteroscedastic weighted least squares
  under the fitted widths (per-point precision $1/(q_i v_i)$), so the
  fit is pinned where the ellipsoid pinches and relaxed where the
  misspecification width is large. Pass `optimize_center=False` to
  freeze the mean at the POPS/BayesianRidge pre-fit and optimize only
  the widths — at small N this is also the more conservative choice.

### Closed-form PAC-Bayes layer

With `pac_bayes=True` the fit adds a diagonal Laplace hyperposterior,
giving closed-form KL and PAC-bound components — still without any
sampling — and the predictive uncertainty gains the analytic
hyperposterior spread.

With the default `hyperprior_center='phase1'` the hyperprior is centered
on the phase-1 optimum itself, so `pac_bayes=True` **never changes the
fitted ellipsoid** (`coef_`, `U_` are identical to the bare fit) and the
predictive bounds are **strictly broader at every point** — the spread
is the epistemic uncertainty *about the ellipsoid*, largest at N/P ~ 2
and decaying at rate N as the hyperposterior concentrates on the bare
values.

`hyperprior_scale` is *relative*: the effective hyperprior variance is
`hyperprior_scale * ||psi_0||^2 / d`, so the default is independent of
the units of `y`. `hyperprior_center='warm_start'` instead centers the
prior on the POPS warm start (the fit is then ridge-shrunk toward the
baseline ellipsoid) and enables the Tipping/MacKay evidence update
`update_hyperprior=True`.

In [ ]:
for N in [10, 50, 500]:
    X_train, x_train, y_train, X_dense, x_dense, y_dense = generate_data(N)
    bare = POPSRegressionEllipse(random_state=0).fit(X_train, y_train)
    pac = POPSRegressionEllipse(random_state=0, pac_bayes=True)
    pac.fit(X_train, y_train)

    _, b_max, b_min = bare.predict(X_dense, return_bounds=True)
    _, p_max, p_min = pac.predict(X_dense, return_bounds=True)
    broadening = np.mean((p_max - p_min) / (b_max - b_min) - 1.0)

    d = pac.hyper_sigma_diag_.size
    print(
        f"N={N:<4d} coverage={pac.coverage_fraction_:.2f}  "
        f"G_hat={pac.objective_:6.3f}  kl/N={pac.kl_ / len(y_train):5.2f}  "
        f"bound={pac.bound_:6.3f}  gamma={pac.gamma_:5.1f} of {d}  "
        f"broadening=+{100 * broadening:.1f}%"
    )

The bound tightens monotonically with N while every training point stays
covered, and the broadening over the bare bounds — strictly positive by
construction — decays as the hyperposterior concentrates on the bare
values at rate N: exactly the behaviour the hierarchical PAC-Bayes
construction prescribes.

### Low-N conservatism: freeze the center

The bare ellipse is an empirical-risk minimizer — the tightest ellipsoid
covering the training residuals — so at N/P ~ 2 it is deliberately much
tighter than the hypercube. The PAC layer broadens it strictly, but with
the jointly-optimized center much of the misspecification is absorbed by
the mean (a heteroscedastic WLS fit), limiting how much width the
hyperposterior can restore.

The conservative low-N configuration is simply
**`optimize_center=False, pac_bayes=True`**: the mean stays at the
POPS/BayesianRidge pre-fit, the widths are optimized by the
interior-point barrier, and the hyperposterior spread — the epistemic
uncertainty about the ellipsoid — is added analytically on top. On a
10-seed sweep of this example at N = 10, that recipe covers 0.99 (mean)
/ 0.91 (worst case) of the dense truth at roughly *60%* of the
hypercube's width, vs 0.78 / 0.64 for the hypercube itself.

In [ ]:
np.random.seed(42)
X_train, x_train, y_train, X_dense, x_dense, y_dense = generate_data(10)

variants = [
    ("POPS Hypercube",
     POPSRegression(leverage_percentile=0.0), "0.5"),
    ("Ellipse (bare)",
     POPSRegressionEllipse(random_state=0), "0.5"),
    ("Ellipse PAC (phase1 center)",
     POPSRegressionEllipse(random_state=0, pac_bayes=True), "0.5"),
    ("Ellipse PAC (warm start + ev.)",
     POPSRegressionEllipse(random_state=0, pac_bayes=True,
                           hyperprior_center="warm_start",
                           update_hyperprior=True), "0.5"),
    ("Frozen center",
     POPSRegressionEllipse(random_state=0, optimize_center=False), "0.5"),
    ("Frozen center + PAC",
     POPSRegressionEllipse(random_state=0, optimize_center=False,
                           pac_bayes=True), "C2"),
]

fig, axs = plt.subplots(2, 3, figsize=(10.5, 6.2), sharex=True, sharey=True)
interp = np.abs(x_dense) <= 10.0
for ax, (name, model, color) in zip(axs.ravel(), variants):
    model.fit(X_train, y_train)
    y_pred, y_max, y_min = model.predict(X_dense, return_bounds=True)
    half_width = np.mean(0.5 * (y_max - y_min)[interp])
    coverage = np.mean(((y_dense >= y_min) & (y_dense <= y_max))[interp])
    ax.fill_between(x_dense, y_min, y_max, alpha=0.3, facecolor=color)
    ax.plot(x_dense, y_pred, "C1-", lw=2.5)
    ax.plot(x_train, y_train, "b.", ms=8)
    ax.plot(x_dense, y_dense, "k-", lw=1)
    ax.set_title(f"{name}\nhalf-width {half_width:.0f}, "
                 f"truth coverage {coverage:.2f}", fontsize=10)
    ax.set_ylim(-260, 260)
    ax.set_xlim(-10, 10)
plt.suptitle("Conservatism test at N = 10 (N/P = 2): bounds vs truth",
             fontsize=12, y=0.99)
plt.tight_layout()
plt.show()